In [ ]:
!pip install sentence-transformers>=3.0.0
!pip install rank_bm25
!pip install pyvi
!pip install faiss-gpu 
print("DONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 85.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 13.7 MB/s eta 0:00:0000:0100:01
DONE


In [2]:
import os
import json 
import zipfile 
import time 
import glob 
import re 
import gc
import torch 
import numpy as np 

WORK_URL   = "/kaggle/working"
TRAIN_URL  = "/kaggle/input/datasets/thngtrnquc/uit-dsc/train.json"
WARMUP_URL = "/kaggle/input/datasets/thngtrnquc/uit-dsc/warmup_IR.json"
PUBLIC_URL = "/kaggle/input/datasets/thngtrnquc/uit-dsc/public-official.json"
CORPUS_FILES_URL = glob.glob("/kaggle/input/**/context_*.json", recursive = True)
if not CORPUS_FILES_URL: 
    raise FileNotFoundError("Không tìm thấy đường dẫn hoặc context_*.json trong kaggle/input")

print("TRAIN:", TRAIN_URL)
print("WARMUP:", WARMUP_URL)
print("PUBLIC:", PUBLIC_URL)
print("CORPUS:", os.path.dirname(CORPUS_FILES_URL[0]), "|", len(CORPUS_FILES_URL), "files")

EMB_MODEL    = "AITeamVN/Vietnamese_Embedding_v2"
RERANK_MODEL = "AITeamVN/Vietnamese_Reranker"

# Hypermeter 
CHUNK_WORDS    = 220 
CHUNK_OVERLAP  = 40 
DENSE_TOPK     = 100 
BM25_TOPK      = 100 
RRF_K          = 60
DOC_CANDIDATES = 30
FINAL_K        = 5 
EMB_BATCH      = 256
RERANK_BATCH   = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs", torch.cuda.device_count())

TRAIN: /kaggle/input/datasets/thngtrnquc/uit-dsc/train.json
WARMUP: /kaggle/input/datasets/thngtrnquc/uit-dsc/warmup_IR.json
PUBLIC: /kaggle/input/datasets/thngtrnquc/uit-dsc/public-official.json
CORPUS: /kaggle/input/datasets/thnghtrnquc/uit-dsc/selected-contexts/selected-contexts | 8532 files
Device: cuda | GPUs 2


In [3]:
def load_corpus(files): 
    docs = {}
    for fp in files: 
        with open(fp, encoding = "utf-8") as f: 
            o = json.load(f)
        docs[str(o["id"])] = {"name": o.get("name", ""),"passage": o.get("passage", "")} 
    return docs 
DOCS = load_corpus(CORPUS_FILES_URL)
print(f"{len(DOCS)} văn bản")

8532 văn bản


In [ ]:
DIEU_RE = re.compile(r"(?=Điều\s+\d+[\.\:]?)")

def sliding(words, size, overlap): 
    step = max(1,size - overlap) 
    for i in range(0, len(words), step): 
        seg = words[i:i+size]
        if seg: 
            yield " ".join(seg)
        if i + size >= len(words): 
            break 

def chunk_doc(name, passage, size = CHUNK_WORDS, overlap = CHUNK_OVERLAP): 
    text = re.sub(r"\s+", " ", passage).strip()
    parts = [p.strip() for p in DIEU_RE.split(text) if p.strip()]
    if len(parts) <= 1: 
        parts = list(sliding(text.split(), size, overlap))
    chunks = []
    for p in parts: 
        w = p.split()
        if len(w) <= size*1.5: 
            chunks.append(p)
        else: 
            chunks.extend(sliding(w, size, overlap))
    prefix = (name.replace("-", " ") + " . ") if name else ""
    return [prefix + c for c in chunks] if chunks else [prefix]

chunk_texts, chunk_doc_ids = [], []
for did, d in DOCS.items(): 
    for c in chunk_doc(d["name"], d["passage"]): 
        chunk_texts.append(c) 
        chunk_doc_ids.append(did)
chunk_doc_ids = np.array(chunk_doc_ids)
print(f"{len(chunk_texts)} chunk từ {len(DOCS)} docs")

In [ ]:
from rank_bm25 import BM25Okapi
USE_PYVI = False 
_tok_re = re.compile(r"[0-9a-zà-ỹ/\-]+", re.IGNORECASE | re.UNICODE)
if USE_PYVI: 
    from pyvi import ViTokenizer
    def vi_tokenizer(text): 
        return _tok_re.findall(ViTokenizer.tokenizer(text.lower()))
else: 
    def vi_tokenizer(text): 
        return _tok_re.findall(text.lower())
bm25_corpus = [vi_tokenizer(t) for t in chunk_texts]
bm25 = BM25Okapi(bm25_corpus)
print(f"BM25 index xong (USE_PYVI = {USE_PYVI}")

In [ ]:
import faiss 
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(EMB_MODEL, device = DEVICE)
emb_model.max_seq_length = 256 

CACHE = os.path.join(WORK_URL, "chunk_emb.npy")
if os.path.exists(CACHE): 
    chunk_emb = np.load(CACHE)
    print("Nạp embedding từ cache", chunk_emb.shape)
else: 
    t0 = time.time()
    chunk_emb = emb_model.encode(chunk_texts, batch_size = EMB_BATCH,
                                show_progress_bar = True, convert_to_numpy = True, 
                                normalize_embeddings = True).astype("float32")
    np.save(CACHE, chunk_emb)
    print(f"Embed {len(chunk_texts)} chunk {time.time() -t0:.1f}s, shape = {chunk_emb.shape}")

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print("FAISS ntotal:", index.ntotal)

In [ ]:
def rrf_fuse(dense_ids, bm25_ids, k= RRF_K): 
    score = {}
    for rank, cid in enumerate(dense_ids): 
        score[cid] = score.get(cid, 0) + 1/(k+rank)
    for rank, cid in enumerate(bm25_ids): 
        score[cid] = score.get(cid, 0) + 1/(k+rank)
    return score 

def retrieve_doc_candidate(query, n_docs = DOC_CANDIDATES): 
    qv = emb_model.encode([query], normalize_embeddings = True, 
                         convert_to_numpy = True).astype("float32")
    _, dense_idx = index.search(qv, DENSE_TOPK)
    dense_idx = dense_idx[0]
    bm25_scores = bm25.get_scores(vi_tokenizer(query))
    bm25_idx = np.argpartition(bm25_scores, -BM25_TOPK)[-BM25_TOPK:]
    bm25_idx = bm25_idx[np.argsort(bm25_scores[bm25_idx])[::-1]]
    fused = rrf_fuse(list(dense_idx), list(bm25_idx))
    # Gom best chunk
    doc_best = {}
    for cid, s in fused.items(): 
        did = chunk_doc_ids[cid]
        if did not in doc_best or s > doc_best[did][1]:
            doc_best[did] = (cid, s)
    ranked = sorted(doc_best.items(), key = lambda x: x[1][1], reverse = True)[:n_docs]
    return [(did, cid) for did, (cid, s) in ranked] # (doc_id, chunk đại diện)

In [ ]:
from sentence_transformers import CrossEncoder 
reranker = CrossEncoder(RERANK_MODEL, device = DEVICE, max_length = 512 )

def rerank_top5 (query, candidates): 
    if not candidates: 
        return []
    pairs = [[query, chunk_texts[cid]] for _, cid in candidates]
    scores = reranker.predict(pairs, batch_size = RERANK_BATCH, show_progress_bar = True)
    order = np.argsort(scores)[::-1][:FINAL_K]
    return [candidates[i][0] for i in order] # 5 doc_id

def predict(query): 
    return rerank_top5(query, retrieve_doc_candidate(query))

In [ ]:
# --- Submission --- 
public = json.load(open(PUBLIC_URL, encoding = "utf-8"))
sub = {}
t0 = time.time()
for i, (qid, it) in enumerate(public.items()): 
    sub[qid] = {"answer": predict(it["question"])}
    if (i+1)%100 ==0: 
        print(f"{i+1}/{len(public)}")
sj = os.path.join(WORK_URL, "submission.json")
json.dump(sub, open(sj, "w", encoding = "utf-8"), ensure_ascii = False)
with zipfile.ZipFile(os.path.join(WORK_URL, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: 
    z.write(sj, "submission.json")
print("DONE")